## events_clean — Validation & Sampling

In [3]:
# ============================================================
# Cell 1: Imports, Auth, Config
# ============================================================
import os
import time
import pandas as pd
from google.oauth2 import service_account
from google.cloud import bigquery

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

# ── Config ───────────────────────────────────────────────────
PROJECT_ID = "recosys-489001"
DATASET_ID = "recosys"
TABLE_ID   = "events_clean"
TABLE_REF  = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
KEY_PATH   = os.path.expanduser(r"C:\Users\Patron\Documents\GitHub\RecoSys\secrets\recosys-service-account.json")

# ── Auth ─────────────────────────────────────────────────────
credentials = service_account.Credentials.from_service_account_file(
    KEY_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
bq = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f"✅ Connected — {credentials.service_account_email}")
print(f"   Table    : {TABLE_REF}")

✅ Connected — 921967012784-compute@developer.gserviceaccount.com
   Table    : recosys-489001.recosys.events_clean


In [4]:
# ============================================================
# Cell 2: Query Helper  (mirrors 03_EDA_BigQuery.ipynb)
# ============================================================
_cache: dict = {}

def run_query(sql: str, label: str = "", use_cache: bool = True) -> pd.DataFrame:
    """
    Run a BigQuery SQL query and return a DataFrame.

    - Dry-runs first to print bytes that will be scanned.
    - Caches results in memory by label (set use_cache=False to force refresh).
    - Prints elapsed time on completion.
    """
    cache_key = label or sql[:120]

    if use_cache and cache_key in _cache:
        df = _cache[cache_key]
        print(f"[cache] '{label}' → {len(df):,} rows")
        return df

    # ── Dry run: estimate cost before executing ──────────────
    dry_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    dry_job    = bq.query(sql, job_config=dry_config)
    bytes_scanned = dry_job.total_bytes_processed
    gb_scanned    = bytes_scanned / 1e9
    cost_est      = (bytes_scanned / 1e12) * 5  # $5 per TB

    print(f"{'─'*60}")
    if label:
        print(f"  Query  : {label}")
    print(f"  Scan   : {gb_scanned:.2f} GB  (~${cost_est:.4f})")
    if gb_scanned > 10:
        print(f"  ⚠️  Large scan — confirm this is intentional")

    # ── Execute ──────────────────────────────────────────────
    t0      = time.time()
    job     = bq.query(sql)
    df      = job.result().to_dataframe()
    elapsed = time.time() - t0

    print(f"  Rows   : {len(df):,}")
    print(f"  Time   : {elapsed:.1f}s")
    print(f"{'─'*60}")

    if label:
        _cache[cache_key] = df
    return df


def clear_cache():
    _cache.clear()
    print("Cache cleared.")

In [5]:
# ============================================================
# Cell 3: events_clean — Full Validation
#
# Single query; two CTE passes over the table:
#   pass 1 (base)  — all scalar metrics
#   pass 2 (bots)  — per-user daily-activity aggregation
# Results are joined in the final SELECT so BQ returns one row.
# ============================================================

VALIDATION_SQL = f"""
WITH

-- ── Pass 1: scalar metrics ──────────────────────────────────
base AS (
  SELECT
    COUNT(*)                          AS total_rows,
    COUNT(DISTINCT user_id)           AS unique_users,
    COUNT(DISTINCT product_id)        AS unique_items,
    COUNTIF(price < 1.0)              AS price_below_1,
    COUNTIF(user_session IS NULL)     AS null_user_session,
    COUNTIF(category_code IS NULL)    AS null_category_code,
    COUNTIF(brand IS NULL)            AS null_brand,
    COUNTIF(event_type = 'view')      AS views,
    COUNTIF(event_type = 'cart')      AS carts,
    COUNTIF(event_type = 'purchase')  AS purchases,
    MIN(event_time)                   AS min_event_time,
    MAX(event_time)                   AS max_event_time
  FROM `{TABLE_REF}`
),

-- ── Pass 2: bot detection — users with avg events/day > 300 ─
daily_activity AS (
  SELECT
    user_id,
    DATE(event_time) AS event_date,
    COUNT(*)         AS daily_events
  FROM `{TABLE_REF}`
  GROUP BY user_id, event_date
),

bot_users AS (
  SELECT COUNT(*) AS bot_user_count
  FROM (
    SELECT user_id
    FROM   daily_activity
    GROUP  BY user_id
    HAVING AVG(daily_events) > 300
  )
)

SELECT
  base.*,
  bot_users.bot_user_count
FROM base, bot_users
"""

row = run_query(VALIDATION_SQL, label="events_clean_validation").iloc[0]

# ── Helpers ──────────────────────────────────────────────────
def _chk(label, actual, expect_fn, fmt=",d"):
    """Print one PASS/FAIL line. expect_fn(actual) must return True for PASS."""
    status = "PASS ✅" if expect_fn(actual) else "FAIL ❌"
    print(f"  {label:<38}  {actual:{fmt}}   {status}")

W = 66
print()
print("═" * W)
print("  events_clean — VALIDATION REPORT")
print(f"  Table: {TABLE_REF}")
print("═" * W)
print(f"  {'CHECK':<38}  {'ACTUAL':>14}   RESULT")
print("─" * W)

# ── Row / cardinality checks ─────────────────────────────────
_chk("Total rows  (expect ~279,937,243)",
     row.total_rows,
     lambda v: 270_000_000 <= v <= 290_000_000)

_chk("Unique users  (expect ~7,565,157)",
     row.unique_users,
     lambda v: 7_000_000 <= v <= 8_000_000)

_chk("Unique items  (expect ~284,523)",
     row.unique_items,
     lambda v: 260_000 <= v <= 310_000)

# ── Data-quality checks (expect 0) ───────────────────────────
print("─" * W)
_chk("Price < 1.0  (expect 0)",
     row.price_below_1,
     lambda v: v == 0)

_chk("NULL user_session  (expect 0)",
     row.null_user_session,
     lambda v: v == 0)

_chk("NULL category_code  (expect 0)",
     row.null_category_code,
     lambda v: v == 0)

_chk("NULL brand  (expect 0)",
     row.null_brand,
     lambda v: v == 0)

_chk("Bot users avg>300 ev/day  (expect 0)",
     row.bot_user_count,
     lambda v: v == 0)

# ── Event-type breakdown (informational) ─────────────────────
print("─" * W)
total = row.total_rows
for etype, cnt in [("views", row.views), ("carts", row.carts), ("purchases", row.purchases)]:
    pct = cnt / total * 100
    print(f"  Event type — {etype:<10}  {cnt:>14,}   ({pct:.1f}%)")

# ── Date range (informational) ───────────────────────────────
print("─" * W)
print(f"  Date range min  {str(row.min_event_time)[:19]}")
print(f"  Date range max  {str(row.max_event_time)[:19]}")

print("═" * W)

# ── Summary ──────────────────────────────────────────────────
checks = [
    270_000_000 <= row.total_rows      <= 290_000_000,
    7_000_000   <= row.unique_users    <= 8_000_000,
    260_000     <= row.unique_items    <= 310_000,
    row.price_below_1    == 0,
    row.null_user_session == 0,
    row.null_category_code == 0,
    row.null_brand         == 0,
    row.bot_user_count     == 0,
]
passed = sum(checks)
total_checks = len(checks)
summary_icon = "✅" if passed == total_checks else "❌"
print(f"  {summary_icon}  {passed}/{total_checks} checks passed")
print("═" * W)

────────────────────────────────────────────────────────────
  Query  : events_clean_validation
  Scan   : 29.60 GB  (~$0.1480)
  ⚠️  Large scan — confirm this is intentional


c:\Users\Patron\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  Rows   : 1
  Time   : 4.9s
────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════
  events_clean — VALIDATION REPORT
  Table: recosys-489001.recosys.events_clean
══════════════════════════════════════════════════════════════════
  CHECK                                           ACTUAL   RESULT
──────────────────────────────────────────────────────────────────
  Total rows  (expect ~279,937,243)       279,937,243   PASS ✅
  Unique users  (expect ~7,565,157)       7,565,157   PASS ✅
  Unique items  (expect ~284,523)         284,523   PASS ✅
──────────────────────────────────────────────────────────────────
  Price < 1.0  (expect 0)                 0   PASS ✅
  NULL user_session  (expect 0)           0   PASS ✅
  NULL category_code  (expect 0)          0   PASS ✅
  NULL brand  (expect 0)                  0   PASS ✅
  Bot users avg>300 ev/day  (expect 0)    0   PASS ✅
───────────────────────────────────────────────